# 10 最小分散＋VTの優位性は市場環境に依存しないか

## 1. 目的と仮説
「最小分散＋ボラティリティ・ターゲティング（VT）は、環境によらず等ウェイト＋VTを上回る」を検証します。
全期間の点推定が良いこと、各観測局面で上回ること、未知の将来にも成立することは別です。
CAGRとSharpeを別々に比較し、下方リスクも確認します。結論は結果に合わせて限定します。

既存の実データ経路を使う追加研究です。局面や閾値は事後的に設定しており、独立した未見のOOS検証ではありません。

## 2. 理論と数式
$$G_s=\left[\prod_{t=1}^n(1+r_{s,t})\right]^{252/n}-1,\quad
SR_s=\sqrt{252}\frac{\bar r_s}{s(r_s)}.$$
無リスク金利は0、Sharpeの分子は算術平均で、CAGRではありません。
$$d_t=\log(1+r_{MV,t})-\log(1+r_{EW,t}),\quad \Delta\ell=252\bar d.$$
年率対数差はCAGR差とは異なる単位。最小分散の最適化は推定共分散に対する分散最小化で、将来のリターンやSharpe優位を保証しません。

## 3. データ・条件
SPY/TLT/GLDの同じ保存済みYahoo調整価格、USD、2006-01-03〜2026-10-29。
252日推定、月次判断、翌営業日終値約定。新保有の損益はその次の終値からです。
目標年率10%、月次目標上限1.5倍（実現倍率はドリフト）、片道5bps、主条件の預金・借入金利0%。
無上限と仮想金利2/3%、4/6%もCSVで比較済み。データ改訂・3ETFの事後選択の限界があります。

In [ ]:
from pathlib import Path
import hashlib, json, inspect
import numpy as np
import pandas as pd
from IPython.display import display, Image
from quant_ai.market_regimes import (
    GROUPS, LABELS, market_states, pair_summary,
    rolling_comparison, paired_regime_bootstrap,
)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/quant_ai').is_dir())
OUT = ROOT / 'outputs/reports/市場環境別VT直接比較'
SOURCE = ROOT / 'outputs/reports/戦略比較と収益分解'
price_path = ROOT / 'data/processed/spy_tlt_gld_adjusted.parquet'
meta = json.loads(price_path.with_suffix('.json').read_text(encoding='utf-8'))
assert hashlib.sha256(price_path.read_bytes()).hexdigest() == meta['sha256']
prices = pd.read_parquet(price_path)
mv = pd.read_parquet(SOURCE / 'path_cap15_rates0_min_variance.parquet')
ew = pd.read_parquet(SOURCE / 'path_cap15_rates0_equal_weight.parquet')
assert mv.index.equals(ew.index)
pair = pd.DataFrame({'minimum': mv.net_return, 'equal': ew.net_return})
states = market_states(prices).loc[pair.index]
assert states['all'].all()
display(pd.Series(pair_summary(pair), name='同条件の直接比較'))
print('実装:', inspect.getfile(pair_summary))

## 4. 前日までの情報で市場局面を定義
株式・債券トレンドはそれぞれSPY・TLTの252日騰落が正か非正か。
株債相関は252日、株式ボラは63日×√252、閾値は20%。全て1日遅らせます。
TLT上昇を「金利低下」と同一視しません。

局面内の日付は非連続なので、局面別CAGRは**条件付き幾何年率換算**です。
その局面だけ取引する実装の運用成績ではなく、非連続局面の最大DDも解釈しません。
トレンド4群は排他的。他の分類とは重複します。

In [ ]:
# t日の分類はt-1終値まで。当日・未来の情報を使わない実装を追跡できます。
print(inspect.getsource(market_states))
regime_rows = []
for group in GROUPS[1:]:
    result = pair_summary(pair.loc[states[group]])
    regime_rows.append({
        '局面': LABELS[group], '日数': result['observations'],
        '最小分散_幾何年率': result['minimum_cagr'],
        '等ウェイト_幾何年率': result['equal_cagr'],
        '年率差': result['cagr_difference'], 'Sharpe差': result['sharpe_difference'],
    })
display(pd.DataFrame(regime_rows))
calendar = pd.read_csv(OUT / 'calendar_periods.csv')
display(calendar.query("case == 'cap15_rates0'"))

## 5. ローリング期間と統計的不確実性
1/3/5年の重複窓でCAGR・Sharpe差を計算。勝率を独立な試行回数とは扱いません。

両戦略収益と局面ラベルを同じ循環ブロックで再標本化します。
価格や配分は再推定しない、観測経路に条件付きの近似です。
ブロック長21/63/126日の各10000回をスクリプトで保存し、ここでは63日を再計算します。
通常95%区間に加え、9群×3指標＋3局面差＝30比較のBonferroni参考区間を表示します。
全30比較への補正は、過去の未記録の研究・モデル選択全体の補正ではありません。
再標本に局面が現れない場合は欠測とし、有効反復数を記録します。

In [ ]:
for window in (252, 756, 1260):
    rolling = rolling_comparison(pair, window)
    print(window, 'CAGR勝率', (rolling.cagr_difference > 0).mean(),
          'Sharpe勝率', (rolling.sharpe_difference > 0).mean())
boot = paired_regime_bootstrap(pair, states, block_days=63, repetitions=10000)
saved = pd.read_csv(OUT / 'bootstrap.csv').query('block_days == 63')
fields = ['estimate', 'lower_95', 'upper_95', 'family_lower_95', 'family_upper_95']
np.testing.assert_allclose(boot[fields], saved[fields], atol=1e-12, equal_nan=True)
display(boot.query("group == 'all'"))
display(boot.query("metric == 'annual_log_active_contrast'"))
print('63日再標本化結果は保存結果と一致')

## 6. 可視化
以下は既存経路から作成した4図。相対資産の下落は、その期間に最小分散VTが劣後したことを示します。

描画コード：[検証スクリプト](../scripts/check_market_independence.py) の `plot_results`。
計算コード：[market_regimes.py](../src/quant_ai/market_regimes.py)。
グラフ・コードを折りたたんだり非表示にしたりしていません。

In [ ]:
FIGURES = ROOT / 'outputs/figures/市場環境別VT直接比較'
for path in sorted(FIGURES.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))

## 7. 統計的解釈
全期間の点推定では最小分散VTのCAGRは10.46%、等ウェイトVTは9.08%、差は+1.37ポイント。
しかし63日ブロックの通常95%区間は約−0.92〜+3.70ポイントで0を跨ぎ、他のブロック長でも同様です。
Sharpe差の区間も0を跨ぎます。全期間の正の差にも、強い統計的裏付けがあるとは言えません。
株/債トレンド・相関の局面差の区間も0を跨ぎ、母集団での環境依存を確定したわけでもありません。

## 8. 金融工学・運用上の解釈
前日SPYボラ20%超の1123日では、最小分散の条件付き幾何年率3.99%に対し等ウェイトは5.35%、Sharpeも低い。
暦期間2006〜2012年、2020〜2021年などでも逆転があります。
全期間の最大DDも最小分散の方が深い。平均リターンだけで一律の優位性を主張しません。
トレンド4群の対数超過収益寄与は株↑・債↓群が大きいですが、これを将来使えるタイミング能力とは呼びません。

## 9. 限界・改善・採用判断
「この過去全期間の平均では上回った」は確認できました。
**「市場環境によらず上回る」は支持できません。** 観測上の逆転はある一方、統計的な依存関係を確定したとの主張も避けます。
目標が同じでも実現ボラは異なるため、完全なリスク同一比較でもありません。
独立将来、他市場、実調達条件、point-in-timeデータ、推定方法、税・流動性・強制縮小を固定条件で検証する必要があります。
投資信託の採用判断にはこれらが不足しています。

[詳しい日本語レポート](../outputs/reports/市場環境に依存しない優位性の検証.md)
[研究選択のバイアスに関する原論文](https://www.nber.org/papers/w21329)